In [ ]:
# Single-cell Drive -> Colab Runtime -> cloud.medicalstudyzone.com sync uploader (resume + verify + retries)
import os
import sys
import json
import time
import shutil
import traceback
import subprocess
import mimetypes
from pathlib import Path
from datetime import datetime

# ----------------------------
# 0) Install dependencies
# ----------------------------
def _ensure_packages():
    packages = ["requests", "tqdm"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

_ensure_packages()

import requests
from tqdm.auto import tqdm

# ----------------------------
# 1) Helpers
# ----------------------------
def log(msg, level="INFO"):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}] [{level}] {msg}")

def norm_rel(p):
    return p.replace("\\", "/").lstrip("./")

def file_sig(path):
    st = os.stat(path)
    return {"size": int(st.st_size), "mtime": int(st.st_mtime)}

def infer_extension_from_signature(path):
    """Best-effort extension inference for extensionless files to improve remote previewability."""
    try:
        with open(path, "rb") as f:
            head = f.read(64)

        if head.startswith(b"\xFF\xD8\xFF"):
            return ".jpg"
        if head.startswith(b"\x89PNG\r\n\x1a\n"):
            return ".png"
        if head.startswith(b"%PDF"):
            return ".pdf"
        if len(head) >= 12 and head[4:8] == b"ftyp":
            # ISO BMFF container (mp4/mov/m4a). Default to mp4 for web preview compatibility.
            return ".mp4"
        if head.startswith(b"ID3"):
            return ".mp3"
        if head.startswith(b"RIFF") and b"WAVE" in head[:16]:
            return ".wav"
    except Exception:
        return ""
    return ""

def read_json(path, default):
    try:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    except Exception:
        log(f"Failed to read state file: {path}. A fresh state will be created.", "WARN")
    return default

def write_json(path, data):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def parse_entries(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for k in ("data", "entries", "items", "fileEntries"):
            if isinstance(payload.get(k), list):
                return payload[k]
    return []

def resolve_api_base(base_input, headers):
    raw = base_input.strip().rstrip("/")
    # Prefer explicit API routes first so we don't accidentally select HTML app routes.
    candidates = []
    if not raw.endswith("/api"):
        candidates.append(raw + "/api")
    if not raw.endswith("/api/v1"):
        candidates.append(raw + "/api/v1")
    candidates.append(raw)

    # Keep order but remove duplicates.
    deduped = []
    seen = set()
    for c in candidates:
        if c not in seen:
            deduped.append(c)
            seen.add(c)

    tested = []
    for c in deduped:
        url = c + "/drive/file-entries"
        try:
            r = requests.get(url, headers=headers, params={"perPage": 1}, timeout=20)
            content_type = (r.headers.get("content-type") or "").lower()

            # Endpoint must behave like JSON API, not HTML app page.
            payload = None
            json_ok = False
            try:
                payload = r.json()
                json_ok = True
            except Exception:
                json_ok = False

            tested.append((url, r.status_code, content_type, json_ok))

            if r.status_code in (200, 401, 403) and json_ok:
                # For 200, validate payload shape is list/dict.
                if r.status_code == 200 and not isinstance(payload, (list, dict)):
                    continue
                return c
        except Exception as e:
            tested.append((url, str(e)))
    raise RuntimeError(f"Could not resolve API base from input '{raw}'. Tried: {tested}")

def build_remote_index(api_base, headers, per_page=200, max_pages=200):
    all_entries = []
    seen_ids = set()

    for page in range(1, max_pages + 1):
        url = api_base + "/drive/file-entries"
        params = {"perPage": per_page, "page": page}
        r = requests.get(url, headers=headers, params=params, timeout=40)

        if r.status_code >= 400:
            if page == 1:
                raise RuntimeError(f"Remote index fetch failed: {r.status_code} {r.text[:400]}")
            break

        payload = r.json()
        entries = parse_entries(payload)
        if not entries:
            break

        newly_added = 0
        for e in entries:
            eid = e.get("id")
            if eid in seen_ids:
                continue
            seen_ids.add(eid)
            all_entries.append(e)
            newly_added += 1

        if newly_added == 0 or len(entries) < per_page:
            break

    id_map = {e.get("id"): e for e in all_entries if e.get("id") is not None}
    remote_files = {}

    for e in all_entries:
        if e.get("type") == "folder":
            continue
        name = e.get("name")
        if not name:
            continue

        parts = [str(name)]
        pid = e.get("parent_id")
        visited = set()
        while pid and pid in id_map and pid not in visited:
            visited.add(pid)
            parent = id_map[pid]
            pname = parent.get("name")
            if pname:
                parts.insert(0, str(pname))
            pid = parent.get("parent_id")

        rel = norm_rel("/".join(parts))
        remote_files[rel] = {
            "id": e.get("id"),
            "size": e.get("file_size"),
            "name": e.get("name"),
        }

    return remote_files, len(all_entries)

def should_skip(rel, src_sig, state_files, remote_index):
    rec = state_files.get(rel)
    if rec and rec.get("status") == "uploaded":
        if (
            rec.get("size") == src_sig["size"]
            and rec.get("mtime") == src_sig["mtime"]
            and rec.get("target_rel") == rel
        ):
            return True, "state"

    remote = remote_index.get(rel)
    if remote:
        rsize = remote.get("size")
        if rsize is None or int(rsize) == src_sig["size"]:
            return True, "remote"

    return False, ""

def upload_with_retry(
    abs_src,
    rel_path,
    staging_root,
    api_base,
    headers,
    max_retries=5,
    backoff_base=2.0,
    timeout=180,
    parent_id=None,
):
    last_error = None
    remote_id = None
    non_retryable_http = {400, 401, 403, 404, 405, 422}

    for attempt in range(1, max_retries + 1):
        staged = os.path.join(staging_root, rel_path)
        os.makedirs(os.path.dirname(staged), exist_ok=True)

        try:
            # Copy from Drive to runtime first
            shutil.copy2(abs_src, staged)

            data = {"relativePath": rel_path}
            if parent_id is not None:
                data["parentId"] = str(parent_id)

            upload_name = os.path.basename(rel_path)
            upload_mime = mimetypes.guess_type(upload_name)[0] or "application/octet-stream"
            if "." not in upload_name:
                log(
                    f"File has no extension: {upload_name}. Preview may fail on destination unless server can infer type.",
                    "WARN",
                )

            with open(staged, "rb") as f:
                files = {"file": (upload_name, f, upload_mime)}
                r = requests.post(
                    api_base + "/uploads",
                    headers=headers,
                    data=data,
                    files=files,
                    timeout=timeout,
                )

            if r.status_code in (200, 201):
                try:
                    payload = r.json()
                    remote_id = (payload.get("fileEntry") or {}).get("id")
                except Exception:
                    remote_id = None
                return True, remote_id, None

            last_error = f"HTTP {r.status_code}: {r.text[:800]}"
            if r.status_code in non_retryable_http:
                return False, remote_id, last_error

        except Exception as e:
            last_error = f"{type(e).__name__}: {e}"

        finally:
            # Delete runtime copy only (never touch source on Drive)
            try:
                if os.path.exists(staged):
                    os.remove(staged)
            except Exception:
                pass

        if attempt < max_retries:
            sleep_s = min(60.0, backoff_base ** attempt)
            log(f"Retry {attempt}/{max_retries} for {rel_path} in {sleep_s:.1f}s. Reason: {last_error}", "WARN")
            time.sleep(sleep_s)

    return False, remote_id, last_error

def sync_drive_to_msz(
    source_drive_folder,
    api_token,
    base_url_input,
    state_file,
    runtime_staging_root,
    target_subdir_mode="source_parent",
    custom_target_subdir="",
    auto_fix_extensionless=True,
    verify_remote=True,
    max_retries=5,
):
    source_drive_folder = os.path.abspath(source_drive_folder)
    runtime_staging_root = os.path.abspath(runtime_staging_root)

    if not os.path.isdir(source_drive_folder):
        raise FileNotFoundError(f"Source folder does not exist: {source_drive_folder}")
    if not api_token or not api_token.strip():
        raise ValueError("API token is empty.")

    source_parent_name = os.path.basename(os.path.normpath(source_drive_folder))
    if target_subdir_mode == "custom":
        target_subdir = norm_rel(custom_target_subdir.strip()).strip("/")
    else:
        target_subdir = norm_rel(source_parent_name).strip("/")

    if not target_subdir:
        raise ValueError("Target subdirectory name resolved to empty value.")

    headers = {"Authorization": f"Bearer {api_token.strip()}"}

    api_base = resolve_api_base(base_url_input, headers)
    log(f"Using API base: {api_base}")
    log(f"Remote target root subdirectory: {target_subdir}")

    # Build file list from Google Drive source
    all_files = []
    for root, _, files in os.walk(source_drive_folder):
        for fn in files:
            abs_p = os.path.join(root, fn)
            rel = norm_rel(os.path.relpath(abs_p, source_drive_folder))
            all_files.append((abs_p, rel))
    all_files.sort(key=lambda x: x[1].lower())

    if not all_files:
        log("No files found in source folder.", "WARN")
        return

    state = read_json(state_file, {
        "version": 1,
        "source_root": source_drive_folder,
        "api_base": api_base,
        "target_subdir": target_subdir,
        "updated_at": None,
        "files": {}
    })
    if "files" not in state or not isinstance(state["files"], dict):
        state["files"] = {}
    state["source_root"] = source_drive_folder
    state["api_base"] = api_base
    state["target_subdir"] = target_subdir

    remote_index = {}
    if verify_remote:
        try:
            log("Building remote index for verification and skip support...")
            remote_index, remote_total = build_remote_index(api_base, headers)
            log(f"Remote index ready. Entries scanned: {remote_total}, files indexed: {len(remote_index)}")
        except Exception as e:
            log(f"Remote verification unavailable ({e}). Continuing with local state only.", "WARN")
            remote_index = {}

    total = len(all_files)
    skipped = 0
    uploaded = 0
    failed = 0

    log(f"Starting sync. Source files discovered: {total}")

    for abs_src, rel in tqdm(all_files, desc="Syncing", unit="file"):
        rel_for_upload = rel
        src_ext = os.path.splitext(rel)[1]
        if auto_fix_extensionless and not src_ext:
            inferred_ext = infer_extension_from_signature(abs_src)
            if inferred_ext:
                rel_for_upload = rel + inferred_ext
                log(f"Extension fixed: {rel} -> {rel_for_upload}", "INFO")
            else:
                log(f"Could not infer extension for: {rel}. Uploading without extension.", "WARN")

        target_rel = norm_rel(f"{target_subdir}/{rel_for_upload}")
        sig = file_sig(abs_src)
        skip, why = should_skip(target_rel, sig, state["files"], remote_index)

        if skip:
            skipped += 1
            if why == "remote" and target_rel not in state["files"]:
                state["files"][target_rel] = {
                    "status": "uploaded",
                    "size": sig["size"],
                    "mtime": sig["mtime"],
                    "uploaded_at": datetime.utcnow().isoformat() + "Z",
                    "target_rel": target_rel,
                    "source": "remote-verified"
                }
            continue

        ok, remote_id, err = upload_with_retry(
            abs_src=abs_src,
            rel_path=target_rel,
            staging_root=runtime_staging_root,
            api_base=api_base,
            headers=headers,
            max_retries=max_retries,
        )

        if ok:
            uploaded += 1
            state["files"][target_rel] = {
                "status": "uploaded",
                "size": sig["size"],
                "mtime": sig["mtime"],
                "uploaded_at": datetime.utcnow().isoformat() + "Z",
                "target_rel": target_rel,
                "remote_id": remote_id
            }
            remote_index[target_rel] = {"id": remote_id, "size": sig["size"], "name": os.path.basename(target_rel)}
            log(f"Uploaded: {target_rel}")
        else:
            failed += 1
            prev_retries = int(state["files"].get(target_rel, {}).get("retries", 0))
            state["files"][target_rel] = {
                "status": "failed",
                "size": sig["size"],
                "mtime": sig["mtime"],
                "target_rel": target_rel,
                "retries": prev_retries + 1,
                "last_error": err,
                "updated_at": datetime.utcnow().isoformat() + "Z"
            }
            log(f"Failed: {target_rel} | {err}", "ERROR")

        state["updated_at"] = datetime.utcnow().isoformat() + "Z"
        write_json(state_file, state)

    # Cleanup empty runtime dirs
    try:
        for root, dirs, files in os.walk(runtime_staging_root, topdown=False):
            if not dirs and not files:
                os.rmdir(root)
    except Exception:
        pass

    state["updated_at"] = datetime.utcnow().isoformat() + "Z"
    write_json(state_file, state)

    log("Sync complete.")
    log(f"Total: {total} | Uploaded: {uploaded} | Skipped: {skipped} | Failed: {failed}")
    log(f"State file: {state_file}")

# ----------------------------
# 2) Colab Form GUI
# ----------------------------
#@markdown <font size="5"><b>Mount / Unmount Google Drive</b></font>
mount_mode = "mount"  #@param ["mount", "unmount", "force_remount"]
mount_path = ""  #@param {type:"string"}

#@markdown Leave mount_path blank to use the default path (<code>/content/drive</code>).
from google.colab import drive
effective_mount_path = mount_path.strip() or "/content/drive"

if mount_mode == "unmount":
    try:
        drive.flush_and_unmount()
        log("Google Drive unmounted.")
    except Exception as e:
        log(f"Unmount failed: {e}", "WARN")
else:
    drive.mount(effective_mount_path, force_remount=(mount_mode == "force_remount"))
    log(f"Google Drive mounted at: {effective_mount_path}")

#@markdown ---
#@markdown <font size="5"><b>Drive to MedicalStudyZone Sync Settings</b></font>

source_drive_folder = "/content/drive/MyDrive/CoreBTR"  #@param {type:"string"}
target_subdir_mode = "custom"  #@param ["source_parent", "custom"]
custom_target_subdir = "CoreBTR"  #@param {type:"string"}
base_url = "https://cloud.medicalstudyzone.com"  #@param {type:"string"}
token_file = "/content/drive/MyDrive/token"  #@param {type:"string"}
api_token = ""  #@param {type:"string"}
state_file = "/content/drive/MyDrive/.msz_sync_state1.json"  #@param {type:"string"}
runtime_staging_root = "/content/msz_runtime_staging"  #@param {type:"string"}
verify_remote = True  #@param {type:"boolean"}
auto_fix_extensionless = True  #@param {type:"boolean"}
max_retries = 5  #@param {type:"slider", min:1, max:10, step:1}
run_sync = True  #@param {type:"boolean"}

#@markdown If api_token is blank, token_file is used. Runtime files are deleted after each upload attempt.

try:
    token_value = api_token.strip()
    if not token_value:
        token_path = Path(token_file.strip())
        if not token_path.exists():
            raise FileNotFoundError(f"Token file not found: {token_path}")
        token_value = token_path.read_text(encoding="utf-8").strip()
        log(f"Token loaded from: {token_path}")

    if run_sync:
        sync_drive_to_msz(
            source_drive_folder=source_drive_folder.strip(),
            api_token=token_value,
            base_url_input=base_url.strip(),
            state_file=state_file.strip(),
            runtime_staging_root=runtime_staging_root.strip(),
            target_subdir_mode=target_subdir_mode.strip(),
            custom_target_subdir=custom_target_subdir.strip(),
            auto_fix_extensionless=bool(auto_fix_extensionless),
            verify_remote=bool(verify_remote),
            max_retries=int(max_retries),
        )
    else:
        log("run_sync is False. Skipping sync.", "WARN")
except Exception as e:
    log(f"Fatal error: {e}", "ERROR")
    traceback.print_exc()